# 01 — Discovery (DirectoryAgent + SearchAgent)
Part of the Public Webcam Discovery System.

Runs DirectoryAgent and SearchAgent against Tier 1 sources. Inspect and save candidates.jsonl.

In [ ]:
# ── Environment setup (run once) ─────────────────────────────
import subprocess, sys, os
from pathlib import Path

IN_COLAB     = "google.colab" in sys.modules
IN_SAGEMAKER = os.environ.get("SM_TRAINING_ENV") is not None

if IN_COLAB:
    if not Path("webcam-discovery").exists():
        subprocess.run(["git", "clone", "https://github.com/YOUR_ORG/webcam-discovery"], check=True)
    os.chdir("webcam-discovery")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[notebooks]", "-q"], check=True)

from webcam_discovery.config import settings
settings.ensure_dirs()
print("✓ Ready")

## Step 2 — Build environment

Install the package with dev dependencies so the pipeline and tests are available.

In [ ]:
!pip install -e ".[dev]" -q

## Step 3 — Run DirectoryAgent (Tier 1)

Crawls all Tier 1 sources from `SOURCES.md`, resolves direct feed URLs via
`FeedExtractionSkill`, and writes results to `candidates/candidates.jsonl`.

In [ ]:
!python scripts/run_discovery.py --tier 1 --output candidates/candidates.jsonl

## Step 4 — Inspect candidates.jsonl

Preview the first few results and count totals.

In [ ]:
import json
from pathlib import Path

output_path = Path("candidates/candidates.jsonl")

if not output_path.exists():
    print("candidates.jsonl not found — did the crawler run successfully?")
else:
    lines = output_path.read_text().splitlines()
    candidates = [json.loads(l) for l in lines if l.strip()]
    print(f"Total candidates: {len(candidates)}")
    print()
    for c in candidates[:10]:
        print(f"  url:     {c['url']}")
        print(f"  label:   {c.get('label', '—')}")
        print(f"  city:    {c.get('city', '—')}, {c.get('country', '—')}")
        print(f"  source:  {c.get('source_directory', '—')}")
        print()